# Day 32: Graph RAG Prototype

## Core Theory (Just-in-Time)

Welcome to Day 32! Today we are diving into **Graph RAG (Retrieval-Augmented Generation)**. 
Traditional RAG relies on semantic similarity using vector embeddings. While great for finding relevant text, it struggles with complex, multi-hop queries where relationships between entities are key (e.g., "Which companies acquired startups funded by Sequoia in 2023?").

Graph RAG solves this by extracting **Entities** (nodes) and **Relationships** (edges) from unstructured text to build a **Knowledge Graph**. When a user asks a question, we query the graph to find connected entities and use the structured context to augment the LLM's response.

### Why Graph RAG?
- **High Precision:** Reduces hallucinations by grounding answers in explicit, structured relationships.
- **Complex Reasoning:** Enables multi-hop reasoning across disparate documents.
- **Explainability:** You can trace exactly which relationships were used to generate an answer.

### How it Works
1. **Extraction:** Use an LLM to parse text and extract entities (e.g., Person, Organization) and their relationships (e.g., "WORKS_FOR", "FOUNDED").
2. **Construction:** Store these nodes and edges in a graph structure (or graph database).
3. **Retrieval:** Traverse the graph to find relevant subgraphs based on the user's query.
4. **Generation:** Pass the retrieved subgraph to the LLM to generate the final answer.


## Code Implementation

We will use LangChain and Pydantic to extract entities and relationships from text. In a production system, you would store these in a graph database like Neo4j, but for this prototype, we will build an in-memory graph using NetworkX.

First, let's define our data models and extraction logic using strict type hinting and docstrings.


In [1]:
import os
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
# Note: You need an OpenAI API key set in your environment: os.environ["OPENAI_API_KEY"] = "your-key"

class Entity(BaseModel):
    """Represents an entity extracted from text."""
    name: str = Field(..., description="The name of the entity, capitalized.")
    type: str = Field(..., description="The type of the entity (e.g., PERSON, ORGANIZATION, LOCATION).")

class Relationship(BaseModel):
    """Represents a relationship between two entities."""
    source: str = Field(..., description="The name of the source entity.")
    target: str = Field(..., description="The name of the target entity.")
    relation_type: str = Field(..., description="The type of relationship (e.g., FOUNDED, ACQUIRED, WORKS_FOR).")

class KnowledgeGraph(BaseModel):
    """A collection of entities and relationships."""
    entities: List[Entity] = Field(default_factory=list, description="List of extracted entities.")
    relationships: List[Relationship] = Field(default_factory=list, description="List of extracted relationships.")

def extract_knowledge_graph(text: str, model_name: str = "gpt-4o-mini") -> KnowledgeGraph:
    """
    Extracts entities and relationships from a given text using an LLM.
    
    Args:
        text (str): The input text to process.
        model_name (str): The LLM model to use for extraction.
        
    Returns:
        KnowledgeGraph: A structured object containing extracted nodes and edges.
    """
    llm = ChatOpenAI(model=model_name, temperature=0.0)
    
    # We use structured output to enforce the KnowledgeGraph schema
    structured_llm = llm.with_structured_output(KnowledgeGraph)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert data extractor. Your task is to extract entities and relationships from the text and build a knowledge graph. Be precise and concise."),
        ("human", "Extract information from the following text:\n\n{text}")
    ])
    
    chain = prompt | structured_llm
    
    result = chain.invoke({"text": text})
    return result

# Example Usage
if __name__ == "__main__":
    sample_text = """
    In 2015, Sam Altman and Elon Musk co-founded OpenAI in San Francisco. 
    Microsoft later invested heavily in OpenAI, which is known for creating ChatGPT.
    """
    try:
        kg = extract_knowledge_graph(sample_text)
        print("Extracted Entities:")
        for entity in kg.entities:
            print(f"- {entity.name} ({entity.type})")
            
        print("\nExtracted Relationships:")
        for rel in kg.relationships:
            print(f"- {rel.source} --[{rel.relation_type}]--> {rel.target}")
    except Exception as e:
        print(f"Execution skipped or failed (likely due to missing API keys): {e}")


Execution skipped or failed (likely due to missing API keys): Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.


### Building and Querying the Graph

Now that we can extract the graph, let's visualize and query it. We'll use `networkx` for our in-memory graph operations.


In [2]:
import networkx as nx

class GraphRAGPrototype:
    """An in-memory Knowledge Graph for RAG operations."""
    
    def __init__(self):
        self.graph = nx.DiGraph()
        
    def add_knowledge(self, kg: KnowledgeGraph) -> None:
        """
        Adds entities and relationships to the graph.
        
        Args:
            kg (KnowledgeGraph): The extracted knowledge graph.
        """
        for entity in kg.entities:
            self.graph.add_node(entity.name, type=entity.type)
            
        for rel in kg.relationships:
            self.graph.add_edge(rel.source, rel.target, relation=rel.relation_type)
            
    def get_context_for_entity(self, entity_name: str, depth: int = 1) -> str:
        """
        Retrieves graph context around a specific entity.
        
        Args:
            entity_name (str): The entity to center the search on.
            depth (int): How many hops away to look.
            
        Returns:
            str: A formatted string of relationships to use as LLM context.
        """
        if not self.graph.has_node(entity_name):
            return f"Entity '{entity_name}' not found in the knowledge graph."
            
        # Get ego graph (neighborhood)
        subgraph = nx.ego_graph(self.graph, entity_name, radius=depth)
        
        context_lines = []
        for u, v, data in subgraph.edges(data=True):
            relation = data.get('relation', 'RELATED_TO')
            context_lines.append(f"{u} -> {relation} -> {v}")
            
        if not context_lines:
            return f"Entity '{entity_name}' found, but has no relationships."
            
        return "\n".join(context_lines)

# Example Usage
if __name__ == "__main__":
    if 'kg' in locals():
        prototype = GraphRAGPrototype()
        prototype.add_knowledge(kg)
        
        query_entity = "OpenAI"
        print(f"\nContext for '{query_entity}':")
        context = prototype.get_context_for_entity(query_entity)
        print(context)


## Practical Lab / Homework

**Your Task:**
1. Extend the `extract_knowledge_graph` function to also extract **Properties** (e.g., date founded, investment amount) and attach them to the nodes or edges.
2. Update the Pydantic models to support a `properties` dictionary field.
3. Write a small script that takes a user query (e.g., "Who invested in the company Sam Altman founded?"), uses an LLM to identify the target entity in the query ("Sam Altman"), retrieves the subgraph context, and answers the question using the retrieved context.

*Hint: Use a standard LCEL chain for the final generation step, passing the `get_context_for_entity` output as context.*


In [3]:
# LAB WORK: Implement your solution here

# 1. Update Pydantic Models (Entity and Relationship with properties)

# 2. Update Extraction Logic

# 3. Create the end-to-end QA chain


## Common Pitfalls

When building Graph RAG in production, look out for:

1. **Entity Resolution / Deduplication:** The LLM might extract "Apple", "Apple Inc.", and "Apple Computer" as three different nodes. You need a normalization or entity resolution step (often using embeddings) to merge identical entities.
2. **Schema Drift:** If you don't constrain the LLM (like we did with Pydantic), it will invent hundreds of slightly different relation types (e.g., `WORKS_AT`, `EMPLOYED_BY`, `IS_EMPLOYEE_OF`). Strict typing and ENUMs are essential.
3. **Graph Traversal Explosion:** Doing deep traversal (depth > 2) on a densely connected graph (like highly referenced entities) can pull in too much context, confusing the LLM and blowing up context windows.
4. **Extraction Latency:** Extracting graphs dynamically for every document at query time is too slow. Extraction must be done asynchronously during the ingestion pipeline.
